In [33]:
##ai agent class assignment
##Movie Expert Agent

import openai, json

client = openai.OpenAI()

messages = []

##유저 메모리
user_profile = {
    "favorite_genres": [],
    "watched_movies": []
}

In [34]:
import requests

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"


##/movies에서 인기 영화를 가져옵니다.
def get_popular_movies():
    try:
        response = requests.get(f"{BASE_URL}/movies")
        response.raise_for_status()

        return json.dumps(response.json(), ensure_ascii=False)

    except requests.exceptions.RequestException as e:
        return json.dumps({
            "status": "error",
            "message": str(e)
        }, ensure_ascii=False)

##/movies/:id에서 영화 정보를 가져옵니다.
def get_movie_details(id):
    try:
        response = requests.get(f"{BASE_URL}/movies/{id}")
        response.raise_for_status()

        return json.dumps(response.json(), ensure_ascii=False)

    except requests.exceptions.RequestException as e:
        return json.dumps({
            "status": "error",
            "message": str(e)
        }, ensure_ascii=False)

##/movies/:id/credits에서 출연진 및 제작진을 가져옵니다.
def get_movie_credits(id):
    try:
        response = requests.get(f"{BASE_URL}/movies/{id}/credits")
        response.raise_for_status()
        
        return json.dumps(response.json(), ensure_ascii=False)

    except requests.exceptions.RequestException as e:
        return json.dumps({
            "status": "error",
            "message": str(e)
        }, ensure_ascii=False)

##좋아하는 장르 추가합니다.
def add_favorite_genre(genre):
    if genre not in user_profile["favorite_genres"]:
        user_profile["favorite_genres"].append(genre)

    return json.dumps({
        "status": "success",
        "favorite_genres": user_profile["favorite_genres"]
    })

##영화 시청 기록 추가합니다.
def add_watched_movie(movie_title):
    if movie_title not in user_profile["watched_movies"]:
        user_profile["watched_movies"].append(movie_title)

    return json.dumps({
        "status": "success",
        "watched_movies": user_profile["watched_movies"]
    })

##사용자 프로필 조회합니다.
def get_user_profile():
    return json.dumps(user_profile, ensure_ascii=False)

##/movies/:id/similar에서 유사한 영화를 조회합니다.
def get_similar_movies(id):
    try:
        response = requests.get(f"{BASE_URL}/movies/{id}/similar")
        response.raise_for_status()
        
        return json.dumps(response.json(), ensure_ascii=False)

    except requests.exceptions.RequestException as e:
        return json.dumps({
            "status": "error",
            "message": str(e)
        }, ensure_ascii=False)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_similar_movies": get_similar_movies,
    "add_favorite_genre": add_favorite_genre,
    "add_watched_movie": add_watched_movie,
    "get_user_profile": get_user_profile,
}

In [35]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": """
            Get the list of currently popular movies.

            Use this function when the user asks:
            - What movies are popular now?
            - Popular movies
            - Trending movies
            - 현재 인기 영화
            - 인기 영화 추천
            """,
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": """
            Get detailed information about a movie by its movie ID.

            Use this function when the user asks:
            - What movie is ID 550?
            - Tell me about movie 550
            - 영화 550 정보 알려줘
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": """
            Get cast and crew information for a movie.

            Use this function when the user asks:
            - Who starred in movie 550?
            - Cast of movie 550
            - 출연진 알려줘
            - 감독 알려줘
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "The movie ID."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_favorite_genre",
            "description": """
            Save user's favorite movie genre.

            Example:
            - 나는 액션 영화를 좋아해
            - SF 장르 좋아함
            - 코미디 영화 좋아해
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "genre": {
                        "type": "string"
                    }
                },
                "required": ["genre"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_watched_movie",
            "description": """
            Save a movie already watched by the user.

            Example:
            - 인터스텔라 봤어
            - Fight Club 이미 봤어
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_title": {
                        "type": "string"
                    }
                },
                "required": ["movie_title"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_user_profile",
            "description": "Get saved user preferences",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": """
            Get movies similar to a movie.

            Use this function when:
            - User asks for similar movies
            - Recommend movies like a specific movie
            - 이 영화와 비슷한 영화 추천
            - 비슷한 영화 알려줘
            """,
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "Movie ID"
                    }
                },
                "required": ["id"]
            }
        }
    }
    
]

In [36]:
SYSTEM_PROMPT = """
You are a Personal Movie Recommendation Agent.

You have access to the following tools.

Movie Tools

1. get_popular_movies()
   - Get currently popular movies.

2. get_movie_details(id)
   - Get detailed information about a movie.

3. get_movie_credits(id)
   - Get cast and crew information.

4. get_similar_movies(id)
   - Get similar movies.

Memory Tools

5. add_favorite_genre(genre)
   - Save a user's favorite genre.

6. add_watched_movie(movie_title)
   - Save a movie the user has already watched.

7. get_user_profile()
   - Retrieve the user's saved preferences.

Rules:

1. When a user says they like a genre,
   use add_favorite_genre().

2. When a user says they watched a movie,
   use add_watched_movie().

3. Do NOT recommend movies automatically.

4. Recommend movies ONLY when the user explicitly asks for recommendations.

5. Before recommending movies,
   use get_user_profile().

6. Never recommend movies already watched.

7. Prefer movies matching favorite genres.

8. If movie information is requested,
   use the movie tools.

9. Use conversation history to personalize recommendations.

10. Always answer in Korean.

11. Always answer in plain text.

12. Do not use markdown

13. Do not use:
*, -, #, bullet points, numbered lists.

14. When saving user preferences,
    do not explain that data was saved.

15. Do not say:
    - 저장했습니다
    - 기억했습니다
    - 이 정보는 저장되었습니다

16. Simply continue the conversation naturally.

17. If the user asks for similar movies,
    use get_similar_movies(id).

18. If the user mentions a movie and asks for recommendations,
    use get_similar_movies(id) after finding the movie.
"""

In [37]:
messages.append({
    "role": "system",
    "content": SYSTEM_PROMPT
})

In [ ]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    } 
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            # print(f"Agent: [{function_name}({arguments}) 호출]")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)
            
            result = function_to_run(**arguments)

            print(f"Agent: [{function_name}({arguments}) 호출]")
            # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result,
            })

        call_ai()

    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"Agent: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)


In [38]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화 알려줘
Agent: [get_popular_movies({}) 호출]
Agent: 현재 인기 있는 영화는 다음과 같습니다:

1. **Obsession**
   - 개요: 호화로운 소원이 이루어지는 과정에서 어두운 대가를 치르는 이야기를 담고 있습니다.
   - 개봉일: 2026-05-13
   - 평점: 7.9
   - 포스터: ![Obsession](https://image.tmdb.org/t/p/w780/bRwnj8WEKBCvmfeUNOukJPwB43K.jpg)

2. **Peddi**
   - 개요: 1980년대의 안드라 프라데시에서 한 마을 사람들이 스포츠를 통해 자존심을 방어하는 이야기입니다.
   - 개봉일: 2026-06-03
   - 평점: 6.8
   - 포스터: ![Peddi](https://image.tmdb.org/t/p/w780/kJAJNNBYlbqAcpTDxBNnaILSMTy.jpg)

3. **The Unknown Man**
   - 개요: 한 작가가 영감을 얻기 위해 남프랑스의 해변가에 고립되려는 이야기를 담고 있습니다.
   - 개봉일: 2021-10-16
   - 평점: 8.1
   - 포스터: ![The Unknown Man](https://image.tmdb.org/t/p/w780/4TpBhdaSl5ALHbgeaYOLF8Q3haz.jpg)

4. **Hai Jawani Toh Ishq Hona Hai**
   - 개요: 주인공이 결혼식을 떠나게 되고 새로운 낯선 사랑과 마주치면서 벌어지는 이야기를 다룹니다.
   - 개봉일: 2026-06-04
   - 평점: 5.4
   - 포스터: ![Hai Jawani Toh Ishq Hona Hai](https://image.tmdb.org/t/p/w780/vmlJvz6qVzYgei2V74GvnmcuQfW.jpg)

5. **Michael**
   - 개요: 마이클 잭슨의 삶을 다룬 다큐멘터리 영화로 그의 음악 이외의 삶을 조명합니다.
   - 개봉일